# Run Segger on Xenium Breast Cancer Data (Colab GPU)

This notebook runs [Segger](https://github.com/dpeerlab/segger) — a GNN-based multimodal segmentation method that combines nuclear masks with transcript point clouds — on the Xenium FFPE Human Breast dataset used in the segmentation benchmark.

Segger requires a CUDA GPU (RAPIDS + PyTorch). Run this notebook on Google Colab with a T4/A10 runtime, then download the output `adata_segger.h5ad` and place it in `data/processed/roi/` of the benchmark repo. All downstream analysis scripts will pick it up automatically.

**Runtime:** ~15–30 min on a T4 GPU for training + prediction on ~3.4M transcripts.

## 1. Install dependencies

In [ ]:
# Verify GPU is available
!nvidia-smi

# Install RAPIDS (cudf, cuspatial, cugraph, rmm) — match CUDA version
!pip install --extra-index-url=https://pypi.nvidia.com \
    cudf-cu12 cuspatial-cu12 cugraph-cu12 rmm-cu12 cupy-cuda12x cuml-cu12

# Install PyTorch Geometric
!pip install torch-geometric

# Install Segger from source
!pip install "git+https://github.com/dpeerlab/segger.git"

In [ ]:
import torch
print(f"PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    raise RuntimeError("No GPU detected — switch to a GPU runtime in Colab.")

import segger
print("Segger imported OK")

## 2. Upload Xenium data

Segger needs the raw Xenium bundle directory. Upload these files from `data/raw/`:
- `transcripts.parquet` (~1.8 GB)
- `cell_boundaries.parquet` (~30 MB)
- `nucleus_boundaries.parquet` (~29 MB)
- `experiment.xenium` (~1.6 KB)

**Option A: Google Drive** (recommended for large files)

In [ ]:
# Option A: Mount Google Drive (upload files to Drive first)
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_DIR = '/content/drive/MyDrive/xenium_data'  # adjust path

# Option B: Direct upload (slow for 1.8 GB transcripts.parquet)
# from google.colab import files
# uploaded = files.upload()

# Option C: wget from a hosted URL (fastest if you have one)
# !mkdir -p /content/xenium_data
# !wget -P /content/xenium_data <YOUR_URL>/transcripts.parquet
# etc.

import os
from pathlib import Path

# Set this to wherever your Xenium bundle files are
DATA_DIR = Path("/content/xenium_data")
DATA_DIR.mkdir(exist_ok=True)

# Verify required files
required = ["transcripts.parquet", "cell_boundaries.parquet",
            "nucleus_boundaries.parquet", "experiment.xenium"]
for f in required:
    path = DATA_DIR / f
    if path.exists():
        print(f"  OK: {f} ({path.stat().st_size / 1e6:.1f} MB)")
    else:
        print(f"  MISSING: {f} — upload this file to {DATA_DIR}")

## 3. Run Segger

Segger trains a GNN on the nuclear mask + transcript graph, then predicts cell-transcript assignments. The `--save-anndata` flag outputs an AnnData with per-cell count matrices.

In [ ]:
OUT_DIR = Path("/content/segger_output")
OUT_DIR.mkdir(exist_ok=True)

!python -m segger segment \
    -i {DATA_DIR} \
    -o {OUT_DIR} \
    --prediction-mode nucleus \
    --save-anndata \
    --n-epochs 20

## 4. Inspect and convert output

In [ ]:
import anndata as ad
import numpy as np

# Find the output AnnData
h5ad_files = list(OUT_DIR.rglob("*.h5ad"))
print(f"Found {len(h5ad_files)} .h5ad file(s):")
for f in h5ad_files:
    print(f"  {f} ({f.stat().st_size / 1e6:.1f} MB)")

adata = ad.read_h5ad(h5ad_files[0])
print(f"\nShape: {adata.shape}")
print(f"obs columns: {list(adata.obs.columns)}")
if "X_spatial" in adata.obsm:
    print(f"Spatial coords in obsm['X_spatial']: {adata.obsm['X_spatial'].shape}")

In [ ]:
# Convert to benchmark-compatible format:
# - obs must have centroid_x, centroid_y, area
# - X is cells x genes count matrix

# Extract centroids from obsm['X_spatial'] if present
if "X_spatial" in adata.obsm:
    coords = adata.obsm["X_spatial"]
    adata.obs["centroid_x"] = coords[:, 0]
    adata.obs["centroid_y"] = coords[:, 1]
    print(f"Centroid range: x=[{coords[:, 0].min():.1f}, {coords[:, 0].max():.1f}], "
          f"y=[{coords[:, 1].min():.1f}, {coords[:, 1].max():.1f}]")
else:
    # Try common column names
    for xc in ["centroid_x", "x", "x_centroid", "cell_x"]:
        if xc in adata.obs.columns:
            adata.obs["centroid_x"] = adata.obs[xc]
            break
    for yc in ["centroid_y", "y", "y_centroid", "cell_y"]:
        if yc in adata.obs.columns:
            adata.obs["centroid_y"] = adata.obs[yc]
            break

if "area" not in adata.obs.columns:
    adata.obs["area"] = 0.0

# Verify
assert "centroid_x" in adata.obs.columns, "centroid_x not found — check output format"
assert "centroid_y" in adata.obs.columns, "centroid_y not found — check output format"

n_tx = int(adata.X.sum()) if hasattr(adata.X, 'sum') else 0
print(f"\n{adata.n_obs} cells, {n_tx} transcripts")
print(f"Median tx/cell: {np.median(np.asarray(adata.X.sum(axis=1)).ravel()):.0f}")

In [ ]:
# The benchmark uses a 2mm x 2mm ROI. Segger runs on the full slide,
# so we crop to the ROI and shift coordinates to match the other methods
# (which use ROI-local coords with origin at top-left corner).
#
# ROI bounds in global Xenium coordinates (from docs/dataset.md):
#   x: [5600, 7600] µm,  y: [8000, 10000] µm

X_MIN, X_MAX = 5600, 7600
Y_MIN, Y_MAX = 8000, 10000

cx = adata.obs["centroid_x"]
cy = adata.obs["centroid_y"]
in_roi = (cx >= X_MIN) & (cx <= X_MAX) & (cy >= Y_MIN) & (cy <= Y_MAX)

print(f"Full slide: {adata.n_obs} cells")
print(f"In ROI: {in_roi.sum()} cells")

if in_roi.sum() < adata.n_obs:
    adata = adata[in_roi].copy()
    print(f"Cropped to ROI: {adata.n_obs} cells")
else:
    print("All cells within ROI bounds (Segger may have auto-cropped).")

# Shift to ROI-local coordinates (origin at top-left of ROI)
adata.obs["centroid_x"] = adata.obs["centroid_x"] - X_MIN
adata.obs["centroid_y"] = adata.obs["centroid_y"] - Y_MIN
print(f"Shifted to ROI-local coords: x=[{adata.obs['centroid_x'].min():.1f}, "
      f"{adata.obs['centroid_x'].max():.1f}], y=[{adata.obs['centroid_y'].min():.1f}, "
      f"{adata.obs['centroid_y'].max():.1f}]")

In [ ]:
# Save the benchmark-compatible AnnData
out_path = "/content/adata_segger.h5ad"
adata.write_h5ad(out_path)
print(f"Saved: {out_path} ({Path(out_path).stat().st_size / 1e6:.1f} MB)")
print(f"\nDownload this file and place it in:")
print(f"  segmentation-benchmark/data/processed/roi/adata_segger.h5ad")
print(f"\nThen rerun the pipeline — all scripts detect it automatically.")

In [ ]:
# Download the file
from google.colab import files
files.download(out_path)